In [2]:
%pip install selenium pandas beautifulsoup4 openpyxl requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: c:\Users\Balaji\.pyenv\pyenv-win\versions\3.11.7\python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

OUTPUT_FILE = Path("arxiv_links_power_distribution.txt")
START_URL = "https://arxiv.org/search/?query=%22power+distribution%22&searchtype=all&abstracts=show&order=-announced_date_first&size=50"

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
all_links = set()

try:
    driver.get(START_URL)
    wait = WebDriverWait(driver, 15)

    while True:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/abs/']")))
        for link in driver.find_elements(By.CSS_SELECTOR, "a[href*='/abs/']"):
            href = link.get_attribute("href")
            if href and "/abs/" in href and "pdf" not in href.lower():
                all_links.add(href)

        print(f"Collected {len(all_links)} unique links...")

        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
            next_href = next_button.get_attribute("href")
            if not next_href:
                break
            driver.get(next_href)
            time.sleep(2)
        except NoSuchElementException:
            break

    OUTPUT_FILE.write_text("\n".join(sorted(all_links)) + ("\n" if all_links else ""), encoding="utf-8")
    print(f"Saved {len(all_links)} links to '{OUTPUT_FILE}'")

finally:
    driver.quit()
    print("Browser closed.")


Collected 50 unique links...
Saved 50 links to 'arxiv_links_power_distribution.txt'
Browser closed.


In [5]:
import os
import re
import random
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver

LINKS_FILE = Path("arxiv_links_power_distribution.txt")
PROCESSED_FILE = Path("processed_links.txt")
EXCEL_FILE = Path("arxiv_data.xlsx")
PDF_DIR = Path("pdfs")
MAX_LINKS_TO_PROCESS = 20  # set to a smaller number to limit the batch size

PDF_DIR.mkdir(exist_ok=True)

if not LINKS_FILE.exists():
    raise FileNotFoundError(f"{LINKS_FILE} not found. Run the previous cell first.")

with LINKS_FILE.open("r", encoding="utf-8") as f:
    all_links = [line.strip() for line in f if line.strip()]

if PROCESSED_FILE.exists():
    with PROCESSED_FILE.open("r", encoding="utf-8") as f:
        processed_links = set(line.strip() for line in f if line.strip())
else:
    processed_links = set()

links_to_process = [link for link in all_links if link not in processed_links][:MAX_LINKS_TO_PROCESS]
print(f"Processing {len(links_to_process)} links (max: {MAX_LINKS_TO_PROCESS})")

if not EXCEL_FILE.exists():
    pd.DataFrame(columns=["Title", "Summary", "URL", "Date", "Author", "PDF_Filename"]).to_excel(EXCEL_FILE, index=False)

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--disable-dev-shm-usage")


def safe_text(tag, default=""):
    if tag is None:
        return default
    return tag.get_text(" ", strip=True)


for link in links_to_process:
    driver = None
    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(link)
        time.sleep(random.uniform(2.0, 4.0))

        total_height = driver.execute_script("return document.body.scrollHeight")
        scroll_position = 0
        while scroll_position < total_height:
            scroll_position += random.randint(150, 300)
            driver.execute_script(f"window.scrollTo(0, {scroll_position});")
            time.sleep(random.uniform(0.3, 0.8))
            total_height = driver.execute_script("return document.body.scrollHeight")

        time.sleep(random.uniform(1.0, 2.0))

        soup = BeautifulSoup(driver.page_source, "html.parser")

        title = safe_text(soup.find("h1", class_="title mathjax"))
        if title.startswith("Title:"):
            title = title.replace("Title:", "", 1).strip()

        summary = safe_text(soup.find("blockquote", class_="abstract mathjax"))
        if summary.startswith("Abstract:"):
            summary = summary.replace("Abstract:", "", 1).strip()

        date_text = safe_text(soup.find("div", class_="dateline"))
        date_match = re.search(r"(\d{1,2} \w+ \d{4})", date_text)
        date = date_match.group(1) if date_match else ""

        authors_tag = soup.find("div", class_="authors")
        authors = ", ".join(a.get_text(" ", strip=True) for a in authors_tag.find_all("a")) if authors_tag else ""

        pdf_button = soup.find("a", class_="abs-button download-pdf")
        pdf_url = f"https://arxiv.org{pdf_button['href']}" if pdf_button and pdf_button.get("href") else ""
        pdf_filename = f"{os.path.basename(pdf_url)}.pdf" if pdf_url else ""
        pdf_path = PDF_DIR / pdf_filename

        if pdf_url and not pdf_path.exists():
            response = requests.get(pdf_url, timeout=30)
            response.raise_for_status()
            pdf_path.write_bytes(response.content)

        row = {
            "Title": title,
            "Summary": summary,
            "URL": pdf_url,
            "Date": date,
            "Author": authors,
            "PDF_Filename": pdf_filename,
        }

        df = pd.read_excel(EXCEL_FILE)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_excel(EXCEL_FILE, index=False)

        with PROCESSED_FILE.open("a", encoding="utf-8") as f:
            f.write(link + "\n")

        print(f"✅ Processed: {link}")

    except Exception as e:
        print(f"❌ Error processing {link}: {e}")

    finally:
        if driver is not None:
            driver.quit()

print("📄 Done! All available links processed.")


Processing 20 links (max: 20)
✅ Processed: https://arxiv.org/abs/2607.07250
✅ Processed: https://arxiv.org/abs/2607.08045
✅ Processed: https://arxiv.org/abs/2607.10994
✅ Processed: https://arxiv.org/abs/2607.11429
✅ Processed: https://arxiv.org/abs/2607.11602
✅ Processed: https://arxiv.org/abs/2607.11681
✅ Processed: https://arxiv.org/abs/2607.12518
✅ Processed: https://arxiv.org/abs/2607.17514
✅ Processed: https://arxiv.org/abs/2607.17992
✅ Processed: https://arxiv.org/abs/2607.20580
✅ Processed: https://arxiv.org/abs/2607.25569
✅ Processed: https://arxiv.org/abs/2607.25863
✅ Processed: https://arxiv.org/abs/2607.26002
✅ Processed: https://arxiv.org/abs/2608.00734
✅ Processed: https://arxiv.org/abs/2608.04061
✅ Processed: https://arxiv.org/abs/2608.07436
✅ Processed: https://arxiv.org/abs/2608.07745
✅ Processed: https://arxiv.org/abs/2608.10623
✅ Processed: https://arxiv.org/abs/2608.13188
✅ Processed: https://arxiv.org/abs/2608.15396
📄 Done! All available links processed.
